# Experiment [2] Visual Dashboard: JEPA v1 Screening Matrix

Screen the medium-scale JEPA variants. The main question is which design choices are stable over seeds and robust under S2/time stress.

In [1]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts" / "[2]"
DATA = ROOT / "data"

PALETTE = {
    "surf_jepa_v0": "#2563eb",
    "raw_flattened": "#64748b",
    "raw_stats": "#f97316",
    "surf_jepa_v0_plus_raw_stats": "#16a34a",
    "large_dual_s2": "#2563eb",
    "large_default": "#7c3aed",
    "large_dual_s2_jepa": "#2563eb",
    "large_default_jepa": "#7c3aed",
    "presto": "#db2777",
    "olmoearth": "#059669",
}
PRIORITY_CONDITIONS = ["clean", "sensor_off_s2", "temporal_drop_50", "temporal_drop_70", "s2_off_tdrop50"]
HOLDOUT_ORDER = ["rwanda-ceo", "togo", "togo-eval", "ethiopia", "lem-brazil"]

pd.options.display.max_columns = 80
pd.options.display.max_rows = 80


def standardize_table(df):
    if df.empty:
        return df
    rename = {
        "balanced_acc": "balanced_accuracy",
        "bal_acc": "balanced_accuracy",
        "mean_auc": "auc",
        "mean_f1": "f1",
        "budget": "label_budget",
    }
    return df.rename(columns={k: v for k, v in rename.items() if k in df.columns and v not in df.columns})


def read_csv(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT) if path.is_absolute() and ROOT in path.parents else path}")
        return pd.DataFrame()
    return standardize_table(pd.read_csv(path, **kwargs))


def read_json(path):
    path = Path(path)
    if not path.exists():
        return {}
    return json.loads(path.read_text())


def pct(x):
    return f"{100*x:.1f}%" if pd.notna(x) else ""


def scorecard(df, cols, title="Scorecard", by=None):
    if df.empty:
        print("No data for scorecard")
        return
    work = df.copy()
    if by:
        work = work.set_index(by)
    display(work[cols].style.format({c: "{:.4f}" for c in cols if pd.api.types.is_numeric_dtype(work[c])}).set_caption(title))


def bar_metric(df, x, y, color=None, title=None, barmode="group", text=True, height=460):
    if df.empty:
        print("No data to plot")
        return None
    fig = px.bar(
        df,
        x=x,
        y=y,
        color=color,
        barmode=barmode,
        text_auto=".3f" if text else False,
        color_discrete_map=PALETTE,
        height=height,
        title=title,
    )
    fig.update_layout(template="plotly_white", legend_title_text="", xaxis_title="", yaxis_title=y)
    fig.show()
    return fig


def line_metric(df, x, y, color=None, facet_col=None, title=None, markers=True, height=520):
    if df.empty:
        print("No data to plot")
        return None
    fig = px.line(
        df,
        x=x,
        y=y,
        color=color,
        facet_col=facet_col,
        markers=markers,
        color_discrete_map=PALETTE,
        height=height,
        title=title,
    )
    fig.update_layout(template="plotly_white", legend_title_text="")
    fig.show()
    return fig


def heatmap(df, x, y, z, title=None, height=520, color_continuous_scale="RdYlGn"):
    if df.empty:
        print("No data to plot")
        return None
    pivot = df.pivot_table(index=y, columns=x, values=z, aggfunc="mean")
    fig = px.imshow(
        pivot,
        text_auto=".3f",
        aspect="auto",
        color_continuous_scale=color_continuous_scale,
        height=height,
        title=title,
    )
    fig.update_layout(template="plotly_white", xaxis_title=x, yaxis_title=y)
    fig.show()
    return fig


def priority_average(df, group_cols, metric_cols=("f1", "auc", "balanced_accuracy"), conditions=PRIORITY_CONDITIONS):
    if df.empty:
        return df
    work = df[df["condition"].isin(conditions)].copy() if "condition" in df else df.copy()
    work = work.rename(columns={"balanced_acc": "balanced_accuracy", "mean_auc": "auc", "mean_f1": "f1"})
    available = [c for c in metric_cols if c in work.columns]
    if not available:
        return work.groupby(group_cols, dropna=False).size().reset_index(name="n")
    return work.groupby(group_cols, dropna=False)[available].mean().reset_index()


def normalize_model_name(name):
    text = str(name)
    return text.replace("_seed42", "").replace("_seed7", "").replace("_seed11", "")

RUN_DIR = ARTIFACTS / "cropharvest_jepa_v1_screen"
ANALYSIS = RUN_DIR / "analysis"
print(RUN_DIR)

/Users/akshithchowdary/Developer/Projects/org/abe/surf/artifacts/[2]/cropharvest_jepa_v1_screen


In [2]:
ranking = read_csv(ANALYSIS / "config_ranking.csv")
scorecard(ranking, ["seeds", "priority_f1", "priority_f1_sd", "s2_tdrop50_f1", "s2_tdrop50_auc"], "[2] Config ranking", by="config")
bar_metric(ranking.sort_values("priority_f1"), x="config", y="priority_f1", title="[2] Priority F1 ranking", height=540)

,seeds,priority_f1,priority_f1_sd,s2_tdrop50_f1,s2_tdrop50_auc
config,,,,,
medium_low_lr,3.0000,0.7825,0.0074,0.7516,0.7511
v0_repro,1.0000,0.7776,nan,0.7554,0.7488
hard_robust_low_lr,3.0000,0.7757,0.0093,0.7488,0.7472
v0_low_lr,3.0000,0.7755,0.0079,0.7474,0.7436
full_s2_outage_robust,3.0000,0.7754,0.0077,0.7457,0.7448
no_robust,3.0000,0.7746,0.0076,0.7358,0.7217
no_modality_aug,3.0000,0.7745,0.0098,0.7430,0.7305
no_temporal_aug,3.0000,0.7736,0.0084,0.7415,0.7322
no_doy,3.0000,0.7729,0.0065,0.7406,0.7332


In [3]:
cond = read_csv(ANALYSIS / "priority_condition_f1.csv")
if not cond.empty:
    long = cond.melt(id_vars="config", var_name="condition", value_name="f1")
    heatmap(long, x="condition", y="config", z="f1", title="[2] F1 by priority condition and config", height=650)

In [4]:
# All probe rows, auto-discovered from run folders.
frames = []
for p in sorted(RUN_DIR.glob("*_seed*")):
    df = read_csv(p / "probe_results.csv")
    if df.empty:
        continue
    config, seed = p.name.rsplit("_seed", 1)
    df["config"] = config
    df["seed"] = int(seed)
    frames.append(df)
probe = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(probe.shape)
probe.head()

(1040, 10)


,model,condition,label_budget,n_train_sub,n_test,f1,auc,balanced_accuracy,config,seed
0,surf_jepa_v0,clean,0.01,541,6770,0.783569,0.786848,0.713597,full_s2_outage_robust,11
1,surf_jepa_v0,clean,0.05,2707,6770,0.802710,0.834115,0.751506,full_s2_outage_robust,11
2,surf_jepa_v0,clean,0.10,5415,6770,0.802177,0.843523,0.758440,full_s2_outage_robust,11
3,surf_jepa_v0,clean,0.25,13538,6770,0.811148,0.856015,0.770700,full_s2_outage_robust,11
4,surf_jepa_v0,clean,1.00,54152,6770,0.817331,0.863556,0.779272,full_s2_outage_robust,11


In [5]:
# Ablation deltas against medium_low_lr.
priority = priority_average(probe, ["config", "seed"])
by_config = priority.groupby("config")[["f1", "auc", "balanced_accuracy"]].agg(["mean", "std"])
display(by_config)
base = priority[priority["config"].eq("medium_low_lr")][["seed", "f1", "auc"]].rename(columns={"f1": "base_f1", "auc": "base_auc"})
deltas = priority.merge(base, on="seed", how="inner")
deltas["delta_f1_vs_medium"] = deltas["f1"] - deltas["base_f1"]
deltas["delta_auc_vs_medium"] = deltas["auc"] - deltas["base_auc"]
bar_metric(deltas[deltas["config"].ne("medium_low_lr")], x="config", y="delta_f1_vs_medium", color="config", title="[2] Seed-matched F1 delta vs medium_low_lr", text=False)

f1                 auc            \
                           mean       std      mean       std   
config                                                          
full_s2_outage_robust  0.775428  0.007748  0.793506  0.006994   
hard_robust_low_lr     0.775655  0.009337  0.792062  0.008552   
medium_low_lr          0.782514  0.007365  0.801237  0.004840   
no_doy                 0.772898  0.006533  0.786879  0.004645   
no_modality_aug        0.774506  0.009792  0.789481  0.011034   
no_robust              0.774558  0.007611  0.789995  0.008137   
no_temporal_aug        0.773556  0.008431  0.787775  0.005687   
v0_low_lr              0.775488  0.007939  0.792789  0.006933   
v0_repro               0.770751       NaN  0.778920       NaN   

                      balanced_accuracy            
                                   mean       std  
config                                             
full_s2_outage_robust          0.718615  0.004752  
hard_robust_low_lr             0.717933  0.005912  
medium_low_lr                  0.727006  0.003748  
no_doy                         0.714319  0.002950  
no_modality_aug                0.716483  0.007905  
no_robust                      0.716829  0.007069  
no_temporal_aug                0.716062  0.004489  
v0_low_lr                      0.718795  0.005255  
v0_repro                       0.708330       NaN

In [6]:
# Label budget curves for selected configs.
selected = ["medium_low_lr", "hard_robust_low_lr", "no_robust", "no_doy", "full_s2_outage_robust"]
curves = probe[probe["config"].isin(selected) & probe["condition"].isin(PRIORITY_CONDITIONS)].groupby(["config", "condition", "label_budget"])[["f1", "auc"]].mean().reset_index()
line_metric(curves, x="label_budget", y="f1", color="config", facet_col="condition", title="[2] Label-efficiency by config", height=680)

In [7]:
# Training summaries: useful for spotting wasted epochs.
train_rows = []
for p in sorted(RUN_DIR.glob("*_seed*")):
    ts = read_json(p / "train_summary.json")
    if ts:
        config, seed = p.name.rsplit("_seed", 1)
        ts.update({"config": config, "seed": int(seed)})
        train_rows.append(ts)
train = pd.DataFrame(train_rows)
display(train.head())
if not train.empty and "best_epoch" in train:
    fig = px.box(train, x="config", y="best_epoch", points="all", title="[2] Best epoch distribution", height=520)
    fig.update_layout(template="plotly_white", xaxis_title="")
    fig.show()

,best_val_loss,train_seconds,history,config,seed
0,0.015919,346.320493,"[{'epoch': 1, 'train_loss': 0.7160699323371604...",full_s2_outage_robust,11
1,0.019237,399.058734,"[{'epoch': 1, 'train_loss': 0.7649921951470552...",full_s2_outage_robust,42
2,0.032433,374.544877,"[{'epoch': 1, 'train_loss': 0.7879692051145766...",full_s2_outage_robust,7
3,0.017384,398.269697,"[{'epoch': 1, 'train_loss': 0.7268138616173355...",hard_robust_low_lr,11
4,0.021387,417.629905,"[{'epoch': 1, 'train_loss': 0.7700231340196397...",hard_robust_low_lr,42
